# Arabic Word-by-Word Pronunciation Comparison
## Using Teacher Recording as Gold Standard

**Text (with tashkeel):**

أَقْبَلَ فَصْلُ الرَّبِيعِ بِحُلَّتِهِ الزَّاهِيَةِ، فَازْدَانَتِ الأَرْضُ بِثَوْبٍ أَخْضَرَ جَمِيلٍ. تَفَتَّحَتِ الزُّهُورُ المُلَوَّنَةُ فِي البَسَاتِينِ، وَانْتَشَرَتْ رَائِحَتُهَا الذَّكِيَّةُ فِي كُلِّ مَكَانٍ. فِي الصَّبَاحِ البَاكِرِ، تُغَرِّدُ الطُّيُورُ عَلَى أَغْصَانِ الأَشْجَارِ كَأَنَّهَا تَعْزِفُ لَحْنًا لِلأَمَلِ. وَتُرْسِلُ الشَّمْسُ أَشِعَّتَهَا الذَّهَبِيَّةَ لِتَنْشُرَ الدِّفْءَ وَالنُّورَ بَيْنَ الرَّوَابِي. يَخْرُجُ الأَطْفَالُ إِلَى الحُقُولِ الواسِعَةِ، يَرْكُضُونَ وَيَمْرَحُونَ بِفَرَحٍ، فَمَا أَجْمَلَ الطَّبِيعَةَ حِينَ تَبْتَسِمُ لَنَا فِي هَذَا الفَصْلِ البَدِيعِ!

---

### Pipeline Overview
1. **Preprocessing**: Convert MP3/M4A → 16kHz mono WAV
2. **Phonemization**: Map tashkeel text to phoneme reference
3. **Forced Alignment**: Use Whisper to locate each word in both recordings
4. **Segmentation**: Cut word-level audio clips
5. **Feature Extraction**: MFCCs + deltas
6. **Comparison**: DTW (Dynamic Time Warping) per word
7. **Scoring**: Duration ratio + acoustic similarity + composite grade
8. **Visualization**: Waveform overlays and score heatmaps

In [ ]:
# Install required packages
# Run this once in your environment
!pip install -q whisper-timestamped librosa soundfile numpy matplotlib fastdtw scipy editdistance

In [ ]:
import re
import json
import os
from pathlib import Path
from collections import namedtuple
from typing import List, Dict, Tuple

import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
import editdistance

import whisper_timestamped

# For phoneme verification (optional, requires ~3GB download)
# from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
# import torch

print("All imports successful")

In [ ]:
# ==============================
# CONFIGURATION
# ==============================

# Input files (update these paths to your actual files)
TEACHER_AUDIO = "./teacher.mp3"      # Teacher recording (gold standard)
STUDENT_AUDIO = "./student.m4a"    # Student recording

# Output cache directory
CACHE_DIR = "./pronunciation_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# Audio settings
TARGET_SR = 16000

# The Arabic text with tashkeel
ARABIC_TEXT = """أَقْبَلَ فَصْلُ الرَّبِيعِ بِحُلَّتِهِ الزَّاهِيَةِ، فَازْدَانَتِ الأَرْضُ بِثَوْبٍ أَخْضَرَ جَمِيلٍ. تَفَتَّحَتِ الزُّهُورُ المُلَوَّنَةُ فِي البَسَاتِينِ، وَانْتَشَرَتْ رَائِحَتُهَا الذَّكِيَّةُ فِي كُلِّ مَكَانٍ. فِي الصَّبَاحِ البَاكِرِ، تُغَرِّدُ الطُّيُورُ عَلَى أَغْصَانِ الأَشْجَارِ كَأَنَّهَا تَعْزِفُ لَحْنًا لِلأَمَلِ. وَتُرْسِلُ الشَّمْسُ أَشِعَّتَهَا الذَّهَبِيَّةَ لِتَنْشُرَ الدِّفْءَ وَالنُّورَ بَيْنَ الرَّوَابِي. يَخْرُجُ الأَطْفَالُ إِلَى الحُقُولِ الواسِعَةِ، يَرْكُضُونَ وَيَمْرَحُونَ بِفَرَحٍ، فَمَا أَجْمَلَ الطَّبِيعَةَ حِينَ تَبْتَسِمُ لَنَا فِي هَذَا الفَصْلِ البَدِيعِ!"""

print(f"Text length: {len(ARABIC_TEXT)} characters")
print(f"Words: {len(ARABIC_TEXT.split())} words")

In [ ]:
# ==============================
# STEP 1: TASHKEEL → PHONEME MAPPER
# ==============================

WordPhoneme = namedtuple('WordPhoneme', ['word', 'phonemes', 'position'])

class ArabicTashkeelPhonemizer:
    """
    Maps Arabic diacritics (tashkeel) to phoneme strings.
    This creates the ground-truth phonetic reference from text.
    """
    
    CONSONANTS = {
        'ا': 'aa', 'ب': 'b', 'ت': 't', 'ث': 'th', 'ج': 'j', 'ح': 'H', 'خ': 'kh',
        'د': 'd', 'ذ': 'dh', 'ر': 'r', 'ز': 'z', 'س': 's', 'ش': 'sh', 'ص': 'S',
        'ض': 'D', 'ط': 'T', 'ظ': 'Z', 'ع': '3', 'غ': 'gh', 'ف': 'f', 'ق': 'q',
        'ك': 'k', 'ل': 'l', 'م': 'm', 'ن': 'n', 'ه': 'h', 'و': 'w', 'ي': 'y',
        'ء': '2', 'ى': 'aa', 'ة': 't', 'ؤ': '2', 'ئ': '2', 'إ': 'i', 'أ': 'a', 'آ': 'aa'
    }
    
    DIACRITICS = {
        '\u064e': 'a',   # Fatha
        '\u064f': 'u',   # Damma
        '\u0650': 'i',   # Kasra
        '\u064b': 'an',  # Fathatan
        '\u064c': 'un',  # Dammatan
        '\u064d': 'in',  # Kasratan
        '\u0651': '~',   # Shadda (doubles consonant)
        '\u0652': '',    # Sukun (no vowel)
        '\u0640': '',    # Tatweel (ignore)
    }
    
    def __init__(self):
        self.diacritic_pattern = re.compile(r'[\u064e\u064f\u0650\u064b\u064c\u064d\u0651\u0652\u0640]')
    
    def phonemize_word(self, word: str) -> str:
        """Convert one Arabic word with tashkeel to phoneme string."""
        phonemes = []
        chars = list(word)
        i = 0
        
        while i < len(chars):
            char = chars[i]
            
            # Skip standalone diacritics (should be attached to previous char)
            if char in self.DIACRITICS:
                if char == '\u0651' and phonemes:
                    phonemes.append(phonemes[-1])  # Double previous consonant
                i += 1
                continue
            
            if char in self.CONSONANTS:
                phonemes.append(self.CONSONANTS[char])
                
                # Check next char for diacritic
                if i + 1 < len(chars) and chars[i+1] in self.DIACRITICS:
                    if chars[i+1] != '\u0651':
                        phonemes.append(self.DIACRITICS[chars[i+1]])
                    i += 1
            i += 1
        
        return ' '.join(phonemes)
    
    def process_text(self, text: str) -> List[WordPhoneme]:
        """Split text into words and generate phonemes."""
        # Remove punctuation except Arabic letters and diacritics
        text = re.sub(r'[^\s\w\u064e\u064f\u0650\u064b\u064c\u064d\u0651\u0652]', '', text)
        text = re.sub(r'\s+', ' ', text.strip())
        words = text.split()
        
        result = []
        for pos, w in enumerate(words):
            if w:
                ph = self.phonemize_word(w)
                result.append(WordPhoneme(word=w, phonemes=ph, position=pos))
        return result

# Generate reference phonemes
phonemizer = ArabicTashkeelPhonemizer()
ref_words = phonemizer.process_text(ARABIC_TEXT)

print(f"Generated phonemes for {len(ref_words)} words\n")
for wp in ref_words[:5]:
    print(f"{wp.position:02d}. {wp.word} → {wp.phonemes}")
print("...")

In [ ]:
# ==============================
# STEP 2: AUDIO PREPROCESSING
# ==============================

class AudioPreprocessor:
    """Normalizes MP3/M4A → 16kHz mono WAV for alignment."""
    
    def __init__(self, target_sr: int = 16000):
        self.target_sr = target_sr
    
    def load_and_convert(self, input_path: str, output_name: str) -> Tuple[np.ndarray, int, str]:
        """
        Load audio file, resample to 16kHz mono, normalize loudness.
        Returns: (audio_array, sample_rate, output_path)
        """
        input_path = Path(input_path)
        output_path = os.path.join(CACHE_DIR, f"{output_name}_16k.wav")
        
        print(f"[Preprocess] Loading: {input_path}")
        
        # librosa uses audioread backend (requires ffmpeg for mp3/m4a)
        audio, sr = librosa.load(str(input_path), sr=self.target_sr, mono=True, res_type='kaiser_best')
        
        # Peak normalization to -1dBFS
        peak = np.max(np.abs(audio))
        if peak > 0:
            audio = audio / peak * 0.891
        
        # Pre-emphasis filter (boosts consonants)
        audio = np.append(audio[0], audio[1:] - 0.97 * audio[:-1])
        
        # Trim silence
        audio, _ = librosa.effects.trim(audio, top_db=40)
        
        sf.write(output_path, audio, self.target_sr, subtype='PCM_16')
        
        duration = len(audio) / self.target_sr
        print(f"[Preprocess] Saved: {output_path} | Duration: {duration:.2f}s\n")
        
        return audio, self.target_sr, output_path

preprocessor = AudioPreprocessor(target_sr=TARGET_SR)

# Process both recordings
teacher_audio, sr, teacher_path = preprocessor.load_and_convert(TEACHER_AUDIO, "teacher")
student_audio, sr, student_path = preprocessor.load_and_convert(STUDENT_AUDIO, "student")

print(f"Teacher duration: {len(teacher_audio)/sr:.2f}s")
print(f"Student duration: {len(student_audio)/sr:.2f}s")
print(f"Duration ratio: {(len(student_audio)/sr) / max(0.001, len(teacher_audio)/sr):.2f}x")

In [ ]:
# ==============================
# STEP 3: FORCED ALIGNMENT (Whisper)
# ==============================

class WordAligner:
    """
    Uses Whisper-Timestamped to find exact word timestamps.
    Model sizes: tiny (39MB), base (74MB), small (244MB), medium (769MB), large (1550MB)
    For Arabic, 'small' or 'medium' recommended.
    """
    
    def __init__(self, model_size="small"):
        print(f"Loading Whisper model: {model_size} (first run downloads weights)...")
        self.model = whisper_timestamped.load_model(model_size)
        print("Model loaded.\n")
    
    def align(self, audio_path: str, text: str) -> List[Dict]:
        """
        Force-align audio to text. Returns word-level timestamps.
        """
        audio = whisper_timestamped.load_audio(audio_path)
        
        result = whisper_timestamped.transcribe_timestamped(
            self.model,
            audio,
            language="ar",
            initial_prompt=text,
            verbose=False
        )
        
        words = []
        for segment in result.get('segments', []):
            for w in segment.get('words', []):
                words.append({
                    'text': w['text'].strip(),
                    'start': w['start'],
                    'end': w['end'],
                    'confidence': w.get('confidence', 0.0)
                })
        return words
    
    def extract_segment(self, audio_path: str, word: Dict) -> Tuple[np.ndarray, int]:
        """Extract audio segment for a specific word."""
        y, sr = librosa.load(audio_path, sr=TARGET_SR, offset=word['start'], duration=word['end']-word['start'])
        return y, sr

# Initialize aligner
aligner = WordAligner(model_size="small")

# Align both recordings to the SAME text
print("Aligning teacher recording...")
teacher_words = aligner.align(teacher_path, ARABIC_TEXT)

print("Aligning student recording...")
student_words = aligner.align(student_path, ARABIC_TEXT)

print(f"\nTeacher words detected: {len(teacher_words)}")
print(f"Student words detected: {len(student_words)}")
print(f"Reference words: {len(ref_words)}")

# Display first few aligned words
print("\nFirst 5 teacher alignments:")
for w in teacher_words[:5]:
    print(f"  {w['text']}: {w['start']:.2f}s → {w['end']:.2f}s")

In [ ]:
# ==============================
# STEP 4: ACOUSTIC COMPARISON (MFCC + DTW)
# ==============================

class AcousticComparator:
    """Compares word audio using MFCC features and Dynamic Time Warping."""
    
    def __init__(self, n_mfcc=13):
        self.n_mfcc = n_mfcc
    
    def extract_features(self, audio: np.ndarray, sr: int) -> np.ndarray:
        """Extract MFCCs with delta and delta-delta features."""
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=self.n_mfcc)
        delta = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)
        return np.vstack([mfcc, delta, delta2]).T  # (time, features)
    
    def compare(self, teacher_audio: np.ndarray, student_audio: np.ndarray, sr: int) -> Dict:
        """
        Compute DTW distance between two audio segments.
        Returns similarity score 0-100 (higher = more similar).
        """
        t_feat = self.extract_features(teacher_audio, sr)
        s_feat = self.extract_features(student_audio, sr)
        
        distance, path = fastdtw(t_feat, s_feat, dist=euclidean)
        
        # Normalize by path length
        norm_dist = distance / max(len(path), 1)
        
        # Convert to similarity (calibrate threshold based on your data)
        similarity = max(0, 100 - (norm_dist * 2.5))
        
        return {
            'dtw_distance': float(distance),
            'normalized_distance': float(norm_dist),
            'similarity_score': float(similarity),
            'path_length': len(path)
        }

# Initialize comparator
comparator = AcousticComparator(n_mfcc=13)

# Run word-by-word comparison
results = []

# Use minimum length to handle cases where alignment counts differ slightly
min_len = min(len(teacher_words), len(student_words), len(ref_words))

print("Running word-by-word comparison...\n")

for i in range(min_len):
    t_w = teacher_words[i]
    s_w = student_words[i]
    ref = ref_words[i]
    
    # Extract audio segments
    t_seg, _ = aligner.extract_segment(teacher_path, t_w)
    s_seg, _ = aligner.extract_segment(student_path, s_w)
    
    # Skip if segments are too short (possible alignment error)
    if len(t_seg) < 400 or len(s_seg) < 400:  # < 25ms at 16kHz
        continue
    
    # Acoustic comparison
    acoustic = comparator.compare(t_seg, s_seg, TARGET_SR)
    
    # Duration analysis
    t_dur = t_w['end'] - t_w['start']
    s_dur = s_w['end'] - s_w['start']
    dur_ratio = s_dur / max(t_dur, 0.001)
    
    # Duration penalty (if student is >2x slower or >2x faster)
    dur_penalty = 0
    if dur_ratio > 2.0 or dur_ratio < 0.5:
        dur_penalty = min(20, abs(1 - dur_ratio) * 10)
    
    results.append({
        'index': i,
        'word': ref.word,
        'phonemes': ref.phonemes,
        'teacher_duration': round(t_dur, 3),
        'student_duration': round(s_dur, 3),
        'duration_ratio': round(dur_ratio, 2),
        'duration_penalty': round(dur_penalty, 1),
        'acoustic_similarity': round(acoustic['similarity_score'], 1),
        'dtw_distance': round(acoustic['dtw_distance'], 2),
        'teacher_text': t_w['text'],
        'student_text': s_w['text']
    })

print(f"Comparison complete for {len(results)} words.\n")

In [ ]:
# ==============================
# STEP 5: COMPOSITE SCORING
# ==============================

def calculate_grade(r: Dict) -> Dict:
    """
    Calculate final pronunciation grade per word.
    Weights: 50% acoustic similarity, 30% duration accuracy, 20% text match
    """
    acoustic = r['acoustic_similarity']
    dur_score = max(0, 100 - r['duration_penalty'] * 5)
    
    # Check if whisper recognized similar text (basic sanity check)
    text_match = 100 if r['teacher_text'] == r['student_text'] else 70
    
    final = (0.5 * acoustic + 0.3 * dur_score + 0.2 * text_match)
    
    if final >= 80:
        status = '✅ EXCELLENT'
    elif final >= 60:
        status = '🟡 GOOD'
    elif final >= 40:
        status = '🟠 NEEDS PRACTICE'
    else:
        status = '🔴 INCORRECT'
    
    return {
        'index': r['index'],
        'word': r['word'],
        'phonemes': r['phonemes'],
        'final_score': round(final, 1),
        'acoustic': round(acoustic, 1),
        'duration_ratio': r['duration_ratio'],
        'status': status
    }

# Generate final report
report = [calculate_grade(r) for r in results]

# Display as formatted table
print(f"{'#':<4} {'Word':<20} {'Phonemes':<25} {'Score':<8} {'Status':<20}")
print("-" * 85)

for item in report[:15]:  # Show first 15
    word_display = item['word'][:18]
    ph_display = item['phonemes'][:23] if item['phonemes'] else 'N/A'
    print(f"{item['index']:<4} {word_display:<20} {ph_display:<25} {item['final_score']:<8} {item['status']:<20}")

if len(report) > 15:
    print(f"\n... and {len(report) - 15} more words\n")

# Overall statistics
scores = [r['final_score'] for r in report]
print(f"\n📊 OVERALL STATISTICS")
print(f"   Total words evaluated: {len(report)}")
print(f"   Average score: {np.mean(scores):.1f}/100")
print(f"   Median score: {np.median(scores):.1f}/100")
print(f"   Excellent (≥80): {sum(1 for s in scores if s >= 80)} words")
print(f"   Needs work (<60): {sum(1 for s in scores if s < 60)} words")

In [ ]:
# ==============================
# STEP 6: VISUALIZATION - Score Heatmap
# ==============================

fig, ax = plt.subplots(figsize=(16, 6))

words_short = [r['word'][:10] for r in report]  # Truncate for display
scores = [r['final_score'] for r in report]
colors = ['#d73027' if s < 40 else '#fc8d59' if s < 60 else '#fee08b' if s < 80 else '#1a9850' for s in scores]

bars = ax.bar(range(len(scores)), scores, color=colors, edgecolor='black', linewidth=0.3)

ax.set_xlabel('Word Index', fontsize=12)
ax.set_ylabel('Pronunciation Score', fontsize=12)
ax.set_title('Word-by-Word Pronunciation Scores: Student vs Teacher', fontsize=14, fontweight='bold')
ax.set_ylim(0, 105)
ax.axhline(y=80, color='green', linestyle='--', alpha=0.5, label='Excellent threshold')
ax.axhline(y=60, color='orange', linestyle='--', alpha=0.5, label='Good threshold')
ax.axhline(y=40, color='red', linestyle='--', alpha=0.5, label='Practice threshold')
ax.legend()

# Add word labels on low-scoring bars
for i, (bar, score, word) in enumerate(zip(bars, scores, words_short)):
    if score < 60:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                word, ha='center', va='bottom', fontsize=7, rotation=45)

plt.tight_layout()
plt.show()

# Save figure
fig.savefig(os.path.join(CACHE_DIR, 'scores_barplot.png'), dpi=150, bbox_inches='tight')
print(f"Saved: {CACHE_DIR}/scores_barplot.png")

In [ ]:
# ==============================
# STEP 7: VISUALIZATION - Duration Comparison
# ==============================

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

indices = [r['index'] for r in results]
t_durs = [r['teacher_duration'] for r in results]
s_durs = [r['student_duration'] for r in results]
ratios = [r['duration_ratio'] for r in results]

# Plot 1: Duration overlay
ax1.plot(indices, t_durs, 'b-', label='Teacher (Gold)', linewidth=2, alpha=0.8)
ax1.plot(indices, s_durs, 'r-', label='Student', linewidth=2, alpha=0.8)
ax1.fill_between(indices, t_durs, s_durs, alpha=0.2, color='purple')
ax1.set_ylabel('Duration (seconds)', fontsize=12)
ax1.set_title('Word Duration Comparison', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Duration ratio
colors_ratio = ['green' if 0.8 <= r <= 1.2 else 'orange' if 0.5 <= r <= 2.0 else 'red' for r in ratios]
ax2.scatter(indices, ratios, c=colors_ratio, s=50, alpha=0.7, edgecolors='black', linewidth=0.5)
ax2.axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label='Perfect match')
ax2.axhline(y=0.5, color='red', linestyle=':', alpha=0.3)
ax2.axhline(y=2.0, color='red', linestyle=':', alpha=0.3)
ax2.set_xlabel('Word Index', fontsize=12)
ax2.set_ylabel('Duration Ratio (Student/Teacher)', fontsize=12)
ax2.set_title('Speed Ratio per Word (1.0 = same speed as teacher)', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

fig.savefig(os.path.join(CACHE_DIR, 'duration_comparison.png'), dpi=150, bbox_inches='tight')
print(f"Saved: {CACHE_DIR}/duration_comparison.png")

In [ ]:
# ==============================
# STEP 8: VISUALIZATION - Waveform Overlay (Single Word)
# ==============================

def plot_word_waveform(word_index: int):
    """Overlay teacher and student waveforms for a specific word."""
    if word_index >= len(results):
        print(f"Invalid index. Max: {len(results)-1}")
        return
    
    r = results[word_index]
    t_w = teacher_words[word_index]
    s_w = student_words[word_index]
    
    t_seg, sr = aligner.extract_segment(teacher_path, t_w)
    s_seg, sr = aligner.extract_segment(student_path, s_w)
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 8))
    
    # Teacher waveform
    t_time = np.linspace(0, len(t_seg)/sr, len(t_seg))
    axes[0].plot(t_time, t_seg, color='blue', alpha=0.8)
    axes[0].set_title(f"Teacher: {r['word']} | Duration: {r['teacher_duration']:.3f}s", fontsize=12)
    axes[0].set_ylabel('Amplitude')
    axes[0].grid(True, alpha=0.3)
    
    # Student waveform
    s_time = np.linspace(0, len(s_seg)/sr, len(s_seg))
    axes[1].plot(s_time, s_seg, color='red', alpha=0.8)
    axes[1].set_title(f"Student: {r['word']} | Duration: {r['student_duration']:.3f}s | Score: {report[word_index]['final_score']:.1f}", fontsize=12)
    axes[1].set_ylabel('Amplitude')
    axes[1].grid(True, alpha=0.3)
    
    # MFCC comparison (heatmap)
    t_mfcc = comparator.extract_features(t_seg, sr)[:, :13].T  # Only MFCCs, no deltas
    s_mfcc = comparator.extract_features(s_seg, sr)[:, :13].T
    
    # Simple side-by-side MFCC heatmap
    ax3 = axes[2]
    im = ax3.imshow(t_mfcc, aspect='auto', origin='lower', cmap='viridis', alpha=0.7, extent=[0, t_time[-1], 0, 13])
    ax3_twin = ax3.twinx()
    ax3_twin.imshow(s_mfcc, aspect='auto', origin='lower', cmap='plasma', alpha=0.5, extent=[0, s_time[-1], 0, 13])
    ax3.set_title('MFCC Overlay (Blue=Teacher, Orange=Student)', fontsize=12)
    ax3.set_xlabel('Time (seconds)')
    ax3.set_ylabel('MFCC Coefficient')
    
    plt.suptitle(f"Word Analysis: {r['word']} [{r['phonemes']}]", fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    fig.savefig(os.path.join(CACHE_DIR, f'word_{word_index:03d}_{r["word"]}.png'), dpi=150, bbox_inches='tight')

# Analyze the lowest scoring word
worst_idx = min(range(len(report)), key=lambda i: report[i]['final_score'])
print(f"Analyzing lowest scoring word (index {worst_idx}): {report[worst_idx]['word']}\n")
plot_word_waveform(worst_idx)

# Also analyze a high-scoring word for comparison
best_idx = max(range(len(report)), key=lambda i: report[i]['final_score'])
print(f"\nAnalyzing highest scoring word (index {best_idx}): {report[best_idx]['word']}\n")
plot_word_waveform(best_idx)

In [ ]:
# ==============================
# STEP 9: EXPORT FULL REPORT
# ==============================

full_report = {
    'metadata': {
        'text': ARABIC_TEXT,
        'teacher_audio': TEACHER_AUDIO,
        'student_audio': STUDENT_AUDIO,
        'total_words': len(report),
        'average_score': round(float(np.mean([r['final_score'] for r in report])), 2),
        'evaluation_date': str(np.datetime64('today'))
    },
    'words': [
        {
            'index': r['index'],
            'word': r['word'],
            'phonemes': r['phonemes'],
            'score': r['final_score'],
            'acoustic_similarity': results[i]['acoustic_similarity'],
            'duration_ratio': results[i]['duration_ratio'],
            'status': r['status'],
            'teacher_duration': results[i]['teacher_duration'],
            'student_duration': results[i]['student_duration']
        }
        for i, r in enumerate(report)
    ],
    'summary': {
        'excellent': sum(1 for r in report if r['final_score'] >= 80),
        'good': sum(1 for r in report if 60 <= r['final_score'] < 80),
        'needs_practice': sum(1 for r in report if 40 <= r['final_score'] < 60),
        'incorrect': sum(1 for r in report if r['final_score'] < 40)
    }
}

output_json = os.path.join(CACHE_DIR, 'pronunciation_report.json')
with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(full_report, f, ensure_ascii=False, indent=2)

print(f"✅ Full report exported to: {output_json}")
print(f"\nSummary:")
print(f"  Excellent:    {full_report['summary']['excellent']} words")
print(f"  Good:         {full_report['summary']['good']} words")
print(f"  Needs Work:   {full_report['summary']['needs_practice']} words")
print(f"  Incorrect:    {full_report['summary']['incorrect']} words")

In [ ]:
# ==============================
# STEP 10: INTERACTIVE WORD LOOKUP
# ==============================

def lookup_word(word_query: str):
    """Find and display all evaluations matching a word."""
    matches = [r for r in report if word_query in r['word']]
    
    if not matches:
        print(f"No matches for '{word_query}'")
        return
    
    print(f"Found {len(matches)} match(es) for '{word_query}':\n")
    for m in matches:
        idx = m['index']
        r = results[idx]
        print(f"─" * 60)
        print(f"Word:      {m['word']}")
        print(f"Phonemes:  {m['phonemes']}")
        print(f"Score:     {m['final_score']}/100 ({m['status']})")
        print(f"Teacher:   {r['teacher_duration']:.3f}s | Student: {r['student_duration']:.3f}s")
        print(f"Ratio:     {r['duration_ratio']:.2f}x")
        print(f"Acoustic:  {r['acoustic_similarity']:.1f}% similarity")
        print(f"DTW Dist:  {r['dtw_distance']:.2f}")
        print(f"─" * 60)

# Example: look up a specific word from the text
lookup_word("الطَّبِيعَة")

# Or lookup by index
# plot_word_waveform(5)

## Notes & Troubleshooting

### Hardware Requirements
| Component | Minimum | Recommended |
|-----------|---------|-------------|
| Whisper (small) | 8GB RAM / 2GB VRAM | 16GB RAM / 4GB VRAM |
| MFCC/DTW | 4GB RAM | 8GB RAM (CPU only) |
| Storage | 2GB | 5GB (model cache) |

### Offline/Air-Gapped Setup
1. **Pre-download Whisper weights** on internet-connected machine:
   ```python
   import whisper_timestamped
   whisper_timestamped.load_model("small")  # Caches to ~/.cache/whisper/
   ```
2. **Transfer cache** to offline server:
   ```bash
   scp -r ~/.cache/whisper/ user@offline-server:/home/user/.cache/
   ```

### Common Issues
- **ffmpeg not found**: Install for MP3/M4A support. On offline RHEL, use static binary from johnvansickle.com/ffmpeg/
- **Alignment mismatch**: If word counts differ between teacher/student, check audio quality and ensure both recordings contain the full text.
- **Low scores on good pronunciation**: Calibrate the `norm_dist * 2.5` scaling factor in `AcousticComparator.compare()` using your dataset.

### Extending This Notebook
- **Phoneme Error Rate (PER)**: Add Wav2Vec2 XLS-R model for phonetic verification layer
- **Pitch analysis**: Use `librosa.pyin()` to compare intonation/melody
- **Tajweed rules**: Extend phonemizer with Idgham, Ikhfa, Ghunnah rules for Quranic text